## Import Necessary Libraries

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
DATA_PATH = "/kaggle/input/datasets/sharduldhekane/code-mixed-hinglish-hate-speech-detection-dataset/combined_hate_speech_dataset.csv"
DATA_PATH2 = "/kaggle/input/datasets/infiniper/hate-speech-detection-dataset/final_cleaned_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(29550, 9)


,text,hate_label,source,profanity_score,language,dataset_version,combined_date,text_length,word_count
0,Knowing ki Vikas kitna samjhata hai Priyanka a...,0,hate_speech_tsv,0,hinglish,v1.0,2025-09-11,126,25
1,I am Muhajir .. Aur mere lye sab se Pehly Paki...,0,hate_speech_tsv,0,hinglish,v1.0,2025-09-11,196,41
2,Doctor sab sahi me ke PhD (in hate politics) ...,0,hate_speech_tsv,0,hinglish,v1.0,2025-09-11,166,29
3,Poore Desh me Patel OBC me aate Hain sirf gujr...,0,hate_speech_tsv,0,hinglish,v1.0,2025-09-11,257,49
4,Sarkar banne ke bad Hindu hit me ek bhi faisla...,1,hate_speech_tsv,0,hinglish,v1.0,2025-09-11,140,25


In [5]:
print(df.columns.tolist())

['text', 'hate_label', 'source', 'profanity_score', 'language', 'dataset_version', 'combined_date', 'text_length', 'word_count']


In [6]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
sample_df = df.sample(20, random_state=42)[["text", "hate_label", "language"]]
sample_df

,text,hate_label,language
10408,ऐतिहासिक रामलीला पर कोरोना का साया टूट जाएगी दो सौ साल पुरानी परंपरा\n\n \n\n,0,hindi
17339,edits i just did another major re do on this article it suffers from creep i do not know who removed all the citations that were in here but please do not do it again remember white sources the article also lost some worldwide refs resulting in a very user centric version i tried to resort a little of info on other nations i encourage editors from outside the united states of america to address more info perhaps here on the talk page or carefully into the article itself but please no links to your farm ok it is very frustrating how many people put in commercial farm links but that said some commercial groups also have some good articles and useful information that while not entirely neutral and not always well sourced also does not appear to be available elsewhere on the web so when the information is accurate please do not remove it just because the link is somewhat commercial unless you can point to a violation of wikipedia policy such as no link farms no links to paid content etc when you can replace a footnote with a more neutral source that is fine i also get the feeling that there are more controversies out there when you edit please be fair to all groups npov is a pillar of wikipedia after all thanks,0,english
15044,and this same teacher will call the germans evil because they taught their school children the same thing about the jews,0,english
17710,america is a free trade country and you know it well impersonator of romanians which call yourself if you you were roman you will put your username roman and not laos aromanians are not greeks and and they cannot be because then they will be greeks and you not romanian i am sorry to hear here people of different mixed nationalities hiding behind a screen impersonating and speaking in name that of the romanians please find true romanian who say whatever are romanian and anything else and ask them and to this nationalist here don't confuse nationality with citizenship aromanians are almost only romanians because of the language and citizens of the countries wherever they live don't be afraid they will ask for parts of your countries and make their own whatever are better where whatever live because whatever are educated and in good positions but which have to hide our whole identity in order to be there and you will never know who are we you just may can whatever can fool you day and night are you angry please don t,1,english
3928,kahenge ye bhi fake hai,1,hinglish
11331,मकान उसने मोके का फ़ायदा उठाया है। बाहरी दिल्ली में ढांग का एक भी हॉस्पिटल दिखा दोना मादरचोदना दिल्ली सरकार कोई हॉस्पिटल बनाता है इसके लिए इलके इलके में प्राइवेट। हॉस्पिटल जनता को लूटने के लिए बने हुए अहिन।,0,hindi
17946,thanks for your explanation but the section in literature etc is in no where are you intended as a list of everything which mentions chopin otherwise it would be endless it is there to give an idea of some of the most significant presentations of chopin which she would some light on his historical life and works the video game sheds light on neither of these it is just apparently a fantasy world in which one of the characters is named chopin and which uses some of chopin s music as arranged by others unless you have a reputable source which says different,0,english
14127,तुम जैसे पुलिस मन समाज के लिए खतरा है,1,hindi
4707,chinaal,1,hinglish
25445,my best friend just sent the and it means cool a k a in his discord and i am disgusted by him especially because i am black,0,english


In [7]:
df = df[["text", "hate_label", "language"]].copy()
df.head()

,text,hate_label,language
0,"Knowing ki Vikas kitna samjhata hai Priyanka aur Itch Guard Luv ko, usne bola tha Ben wali baat me ab Sallu ne bhi agree kiya!",0,hinglish
1,I am Muhajir .. Aur mere lye sab se Pehly Pakistan he .. agr 10 lakh Altaf Jese leaders bh is zameen ki behurmati kren un sbko sar e aam phansi Deni chahye .. Proud to be a #Muhajir and #Pakistani,0,hinglish
2,"Doctor sab sahi me ke PhD (in hate politics) wale. Bhai padhe likhe ho fir kyu ye sab baate karte ho. Tum bas bowling khelo, aur maje lo. pic.twitter.com/fk1qUbQstw",0,hinglish
3,"Poore Desh me Patel OBC me aate Hain sirf gujrat Ko chor kar may be, ye manuwadiyon bramanwadi kabhi aapko aarackchan nahi denge ye to jis OBC Ko Mila hai usse bhi nafrat karte hain ye khoon aur chamdi ka frak karne waale bharmhanwadi kisi ke sage nahi hain",0,hinglish
4,"Sarkar banne ke bad Hindu hit me ek bhi faisla Jo bjp ke dwara liya gaya ho,bjp ko gay,gobar,mandir,masjid aur nafrat faila kar vot chahiye",1,hinglish


## Basic Cleaning

In [8]:
def clean_text(text):
    text = str(text)
    
    text = re.sub(r"http\S+", "", text)   # remove urls
    text = re.sub(r"@\w+", "", text)      # mentions
    text = re.sub(r"#\w+", "", text)      # hashtags
    text = re.sub(r"\s+", " ", text)      # extra spaces
    
    return text.strip()

df["text"] = df["text"].apply(clean_text)

In [9]:
df["text"] = df["text"].apply(clean_text)
df.head()

,text,hate_label,language
0,"Knowing ki Vikas kitna samjhata hai Priyanka aur Itch Guard Luv ko, usne bola tha Ben wali baat me ab Sallu ne bhi agree kiya!",0,hinglish
1,I am Muhajir .. Aur mere lye sab se Pehly Pakistan he .. agr 10 lakh Altaf Jese leaders bh is zameen ki behurmati kren un sbko sar e aam phansi Deni chahye .. Proud to be a and,0,hinglish
2,"Doctor sab sahi me ke PhD (in hate politics) wale. Bhai padhe likhe ho fir kyu ye sab baate karte ho. Tum bas bowling khelo, aur maje lo. pic.twitter.com/fk1qUbQstw",0,hinglish
3,"Poore Desh me Patel OBC me aate Hain sirf gujrat Ko chor kar may be, ye manuwadiyon bramanwadi kabhi aapko aarackchan nahi denge ye to jis OBC Ko Mila hai usse bhi nafrat karte hain ye khoon aur chamdi ka frak karne waale bharmhanwadi kisi ke sage nahi hain",0,hinglish
4,"Sarkar banne ke bad Hindu hit me ek bhi faisla Jo bjp ke dwara liya gaya ho,bjp ko gay,gobar,mandir,masjid aur nafrat faila kar vot chahiye",1,hinglish


## Label Distribution


In [10]:
print(df["hate_label"].value_counts())

hate_label
0    15825
1    13725
Name: count, dtype: int64


## Train-Test-Split

In [11]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["hate_label"]
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)

Train: (23640, 3)
Val: (5910, 3)


In [12]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

## Load Tokenizer

In [13]:
MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

## Tokenization

In [14]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Map:   0%|          | 0/23640 [00:00<?, ? examples/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Map:   0%|          | 0/5910 [00:00<?, ? examples/s]

In [15]:
train_dataset = train_dataset.rename_column("hate_label", "labels")
val_dataset = val_dataset.rename_column("hate_label", "labels")

In [16]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

## load MuRIL Model

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

### Metric Function Custom

In [18]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions)
    }

### Data Collator- basically dynamic padding

In [19]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [20]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["hate_label"]),
    y=df["hate_label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

Class weights: tensor([0.9336, 1.0765])


### Training Arguements

In [21]:
from transformers import Trainer
import torch.nn as nn
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, 2),   # binary classification
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [22]:
from transformers import EarlyStoppingCallback

In [23]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    logging_dir="./logs",
    logging_steps=100,

    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Trainer

In [24]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.590733,0.572677,0.716582,0.690445,0.700675,0.680510
2,0.516853,0.534662,0.726227,0.701696,0.710340,0.693260
3,0.477605,0.525535,0.736210,0.696042,0.748742,0.650273
4,0.425129,0.537869,0.736379,0.699228,0.743737,0.659745


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


...

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2956, training_loss=0.5210893673567714, metrics={'train_runtime': 1818.9651, 'train_samples_per_second': 64.982, 'train_steps_per_second': 2.031, 'total_flos': 5696255681898240.0, 'train_loss': 0.5210893673567714, 'epoch': 4.0})

## Model Evaluation

In [26]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.5340068340301514, 'eval_accuracy': 0.7272419627749577, 'eval_f1': 0.7007055328629781, 'eval_precision': 0.7145020825444908, 'eval_recall': 0.687431693989071, 'eval_runtime': 37.0974, 'eval_samples_per_second': 159.31, 'eval_steps_per_second': 4.987, 'epoch': 4.0}


In [27]:
predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.74      0.76      0.75      3165
           1       0.71      0.69      0.70      2745

    accuracy                           0.73      5910
   macro avg       0.73      0.72      0.73      5910
weighted avg       0.73      0.73      0.73      5910



In [28]:
print(confusion_matrix(labels, preds))

[[2411  754]
 [ 858 1887]]


### Test model

In [29]:
def predict_text(text):
    trainer.model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = trainer.model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

    print("Probabilities:", probs.cpu().numpy())

    return pred

In [30]:
texts = [
    "you are amazing",
    "tum pagal ho",
    "I hate this community",
    'dhat',
    'bhad me jaa',
    'dhee ke lath',
    'gand mar',
    'muthmare',
    'goon',
    'teri maa ka bharosa',
    'teri maa ka bhosra'
]

for t in texts:
    print(t, "->", predict_text(t))

Probabilities: [[0.78862524 0.2113748 ]]
you are amazing -> 0
Probabilities: [[0.69086874 0.30913126]]
tum pagal ho -> 0
Probabilities: [[0.6091654  0.39083466]]
I hate this community -> 0
Probabilities: [[0.14010589 0.85989404]]
dhat -> 1
Probabilities: [[0.72304505 0.27695495]]
bhad me jaa -> 0
Probabilities: [[0.70911115 0.2908888 ]]
dhee ke lath -> 0
Probabilities: [[0.13649543 0.8635045 ]]
gand mar -> 1
Probabilities: [[0.13557325 0.86442673]]
muthmare -> 1
Probabilities: [[0.1369597  0.86304027]]
goon -> 1
Probabilities: [[0.85330576 0.14669429]]
teri maa ka bharosa -> 0
Probabilities: [[0.5385999 0.4614001]]
teri maa ka bhosra -> 0


## save model

In [31]:
trainer.save_model("muril_hate_model")
tokenizer.save_pretrained("muril_hate_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!
The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


('muril_hate_model/tokenizer_config.json', 'muril_hate_model/tokenizer.json')